In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv('../../datasets/IMDB Dataset.csv')
df.shape

(50000, 2)

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
r1 = df.iloc[0,0]
print(r1)

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [5]:
pattern = '<.*?>'
r1 = re.sub(pattern,' ',r1)
print(r1)

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.  The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.  It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.  I would say the main appeal of the show is due to the fact that it goes where other sho

In [6]:
cleaned_reviews = df['review'].apply(lambda r1:re.sub(pattern,' ',r1))

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
tokenizer = Tokenizer(num_words=14000)
tokenizer.fit_on_texts(cleaned_reviews)

In [9]:
max(tokenizer.index_word.keys())

124245

In [10]:
tokenizer.index_word[2101]

'reviewers'

In [11]:
tokenized_reviews = tokenizer.texts_to_sequences(cleaned_reviews)

In [12]:
print((tokenized_reviews[0]))

[26, 4, 1, 78, 2101, 44, 1071, 11, 99, 146, 38, 306, 3183, 397, 473, 25, 3194, 32, 22, 202, 13, 10, 6, 620, 47, 595, 15, 67, 1, 85, 147, 11, 3240, 67, 41, 3183, 12, 92, 5395, 2, 133, 4, 569, 59, 267, 7, 202, 35, 1, 659, 138, 1739, 67, 10, 6, 20, 3, 118, 14, 1, 7887, 2332, 37, 11468, 10, 118, 2594, 53, 5910, 15, 5509, 5, 1478, 375, 37, 569, 92, 6, 3803, 7, 1, 359, 355, 4, 1, 659, 8, 6, 432, 3183, 13, 11, 6, 1, 11647, 357, 5, 1, 6812, 2537, 1063, 8, 2710, 1420, 19, 537, 31, 4635, 2467, 4, 1, 1207, 116, 28, 1, 7016, 24, 2969, 11963, 2, 390, 33, 6, 20, 298, 19, 1, 4909, 7477, 537, 6, 343, 5, 105, 8160, 5049, 7888, 2452, 2, 50, 33, 326, 9104, 7363, 12485, 2, 8696, 22, 109, 224, 242, 9, 57, 130, 1, 279, 1323, 4, 1, 118, 6, 692, 5, 1, 191, 11, 8, 268, 116, 78, 275, 588, 3023, 833, 179, 1319, 4160, 14, 2522, 1242, 833, 1442, 833, 886, 3183, 148, 953, 182, 1, 85, 397, 9, 122, 209, 3240, 67, 13, 33, 1636, 8, 12, 2238, 9, 412, 130, 9, 12, 1591, 14, 8, 17, 13, 9, 286, 50, 9, 1415, 3, 1279, 14, 318

In [13]:
# sum([ len(review) for review in tokenized_reviews])/50000
max([ len(review) for review in tokenized_reviews])

2234

In [14]:
[ len(review) for review in tokenized_reviews[:5]]

[295, 147, 160, 131, 214]

In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [16]:
padded_reviews = pad_sequences(tokenized_reviews,maxlen=220)

In [17]:
[ len(review) for review in padded_reviews[:5]]

[220, 220, 220, 220, 220]

In [18]:
# padded_reviews[1]

In [19]:
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential

In [22]:
y = df['sentiment'].map({'positive':1,'negative':0})

In [23]:
model = Sequential()
model.add(Embedding(input_dim=14000,output_dim=32))
model.add(LSTM(50,return_sequences=False))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',metrics=['accuracy'])
model.fit(padded_reviews,y,batch_size=64,epochs=1)

782/782 ━━━━━━━━━━━━━━━━━━━━ 50s 63ms/step - accuracy: 0.7135 - loss: 0.5247


In [24]:
model.fit(padded_reviews,y,batch_size=64,epochs=1)

782/782 ━━━━━━━━━━━━━━━━━━━━ 49s 63ms/step - accuracy: 0.8843 - loss: 0.2862


In [26]:
test_reviews = ['''Amazing movie with great story''',
           '''I recently watched, and unfortunately, it was a major disappointment. The movie starts with an interesting concept, but the story quickly becomes predictable and boring. The screenplay is poorly written, with several unnecessary scenes that make the movie feel much longer than it actually is. The characters lack depth, and their decisions often seem unrealistic. Even the lead actors, who are usually impressive, fail to deliver convincing performances here. The dialogues feel forced, and the humor rarely works. The background music is repetitive and does little to improve the overall experience. The biggest problem is the weak climax, which leaves many questions unanswered and feels rushed.''',
           '''“I think this movie's rating might be slightly clouded with nostalgia. It is not a bad movie necessarily, but sometimes the plot really makes no sense. Some characters lack any form of motivation for their actions.”''',
           '''From the very first scene, the movie grabs your attention and refuses to let go. It is rare to find a film that balances a deeply moving emotional core with such stunning visual direction.The performances from the lead cast feel entirely genuine, grounding the high-stakes narrative in raw, human moments. Instead of relying on cheap tricks or over-complicated plot twists, the movie trusts its audience, allowing the tension and character arcs to breathe and develop naturally. The cinematography and score work in perfect harmony, creating an atmosphere that stays with you long after the credits roll.It doesn't just entertain; it makes you feel deeply and think about the story's themes. An absolute triumph from start to finish.''',
           '''3 Idiots is a 2009 Hindi-language comedy-drama film that blends sharp humor with a powerful message about following your true passion.Plot and StorylineThe Journey: The story follows two friends searching for their long-lost college buddy, Rancho.The Flashbacks: The film uses flashbacks to show their college days at an elite engineering institute.The Core Trio: Rancho encourages Farhan, who loves photography, and Raju, who carries heavy family pressure, to break free from fear and expectations.Performances and DirectionRajkumar Hirani: The director balances heavy emotional drama and laugh-out-loud comedy with ease.Aamir Khan: He plays Rancho with charm and deep sincerity, leading a talented cast that includes R. Madhavan and Sharman Joshi.Boman Irani: He plays the strict college director, Viru Sahastrabuddhe, delivering a memorable and intense antagonistic presence.Themes and MusicEducation System: The film critiques rote learning and extreme academic pressure, urging students to chase excellence rather than just marks.Soundtrack: Composed by Shantanu Moitra, memorable tracks like "All Izz Well" add energy and warmth to the narrative.''']

In [27]:
tokenized_test_reviews = tokenizer.texts_to_sequences(test_reviews)

In [28]:
padded_test_reviews = pad_sequences(tokenized_test_reviews,maxlen=220)

In [29]:
model.predict(padded_test_reviews)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step


array([[0.92579335],
       [0.01903873],
       [0.18260978],
       [0.97024393],
       [0.9635026 ]], dtype=float32)